In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import optuna

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# =========================
# 1. 读取数据
# =========================
df0 = pd.read_excel("Dataset_DFT.xlsx")
df = df0.copy()

# 如需删除无关列，可取消注释
df.drop(columns=["Type", "DBPs","DOI", "Code", "DBP_group"], inplace=True)


# =========================
# 2. 定义输入和输出
# =========================
target_col = "C"

X = df.drop(columns=[target_col])
y = df[target_col]


# =========================
# 3. 定义需要归一化的列
# =========================
numerical_cols_to_scale = [
    "DOC", "UV254", "Br", "TN", "Chlorine dose", "Disinfection time",
    "Temperature", "pH", "C1", "C2", "LMW", "MMW", "HMW",
    "Es", "EHOMO", "ELUMO", "Egap"
]


# =========================
# 4. 先划分训练集和测试集
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=72
)


# =========================
# 5. 构建预处理器
# 只归一化指定列，其余列保持不变
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numerical_cols_to_scale)
    ],
    remainder="passthrough"
)

# =========================
# 导出完整列，并保持与 train_raw/test_raw 相同的样本顺序
# =========================

# 按 X_train_raw 和 X_test_raw 的实际顺序提取原始完整数据
train_full = df0.loc[X_train_raw.index].copy()
test_full = df0.loc[X_test_raw.index].copy()

# 添加数据集标记
train_full["Dataset_split"] = "Training"
test_full["Dataset_split"] = "Test"

# 先放训练集，再放测试集
dataset_full = pd.concat(
    [train_full, test_full],
    axis=0,
    ignore_index=True
)

dataset_full.to_excel(
    "Dataset_train_test_split.xlsx",
    index=False
)

print("训练集样本数：", len(train_full))
print("测试集样本数：", len(test_full))
print("完整数据已保存至 Dataset_train_test_split.xlsx")

d:\anaconda3\envs\MLenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


训练集样本数： 1896
测试集样本数： 475
完整数据已保存至 Dataset_train_test_split.xlsx


In [26]:
# =========================
# 6. Optuna 贝叶斯优化
# =========================
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.2, log=True),

        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),

        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "tree_method": "hist",

        "grow_policy": trial.suggest_categorical(
        "grow_policy",
        ["depthwise", "lossguide"]
        ),

        "max_leaves": trial.suggest_int("max_leaves", 16, 128),

        # 损失函数
        "objective": trial.suggest_categorical(
         "objective",
        [
            "reg:squarederror",
            "reg:pseudohubererror"
        ]
        ),
        
        "random_state": 72,
        "n_jobs": 1,
        }

    model = XGBRegressor(**params)

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    cv = KFold(
        n_splits=10,
        shuffle=True,
        random_state=72
    )

    scores = cross_val_score(
        pipe,
        X_train_raw,
        y_train,
        scoring="r2",
        cv=cv,
        n_jobs=1
    )

    return np.mean(scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best CV R2:", study.best_value)
print("Best params:", study.best_params)

[I 2026-07-14 16:37:19,082] A new study created in memory with name: no-name-9725b643-6b41-4de6-84bf-5e0b10fc6942
Best trial: 0. Best value: 0.552677:   2%|▏         | 1/50 [00:01<00:51,  1.06s/it]

[I 2026-07-14 16:37:20,143] Trial 0 finished with value: 0.5526767759657071 and parameters: {'n_estimators': 704, 'max_depth': 2, 'learning_rate': 0.006913066988510718, 'subsample': 0.5972339158145419, 'colsample_bytree': 0.8104285821070347, 'colsample_bylevel': 0.5837627112409529, 'colsample_bynode': 0.7782166307873952, 'reg_alpha': 0.19591648104590326, 'reg_lambda': 0.00046822080597917607, 'gamma': 0.7164731265555713, 'min_child_weight': 5, 'grow_policy': 'depthwise', 'max_leaves': 78, 'objective': 'reg:squarederror'}. Best is trial 0 with value: 0.5526767759657071.


Best trial: 1. Best value: 0.680011:   4%|▍         | 2/50 [00:04<01:45,  2.20s/it]

[I 2026-07-14 16:37:23,147] Trial 1 finished with value: 0.6800105987209265 and parameters: {'n_estimators': 1936, 'max_depth': 2, 'learning_rate': 0.10249732128647072, 'subsample': 0.5444720621789949, 'colsample_bytree': 0.7983148503142637, 'colsample_bylevel': 0.8234381383170111, 'colsample_bynode': 0.8875833096023309, 'reg_alpha': 0.0011846079286330533, 'reg_lambda': 0.2614174212138644, 'gamma': 1.9490562684268964, 'min_child_weight': 2, 'grow_policy': 'lossguide', 'max_leaves': 44, 'objective': 'reg:pseudohubererror'}. Best is trial 1 with value: 0.6800105987209265.


Best trial: 1. Best value: 0.680011:   6%|▌         | 3/50 [00:05<01:23,  1.78s/it]

...

[I 2026-07-14 16:41:27,040] Trial 47 finished with value: 0.8618642484364607 and parameters: {'n_estimators': 1302, 'max_depth': 8, 'learning_rate': 0.030396392464356676, 'subsample': 0.6275137280101084, 'colsample_bytree': 0.9461377230267906, 'colsample_bylevel': 0.664848832268635, 'colsample_bynode': 0.9165007149931678, 'reg_alpha': 0.0008725200955613633, 'reg_lambda': 0.0015307261697614304, 'gamma': 0.4942011787573195, 'min_child_weight': 7, 'grow_policy': 'lossguide', 'max_leaves': 85, 'objective': 'reg:squarederror'}. Best is trial 29 with value: 0.8657825026628405.


Best trial: 48. Best value: 0.865944:  98%|█████████▊| 49/50 [04:16<00:08,  8.25s/it]

[I 2026-07-14 16:41:35,410] Trial 48 finished with value: 0.8659439589259709 and parameters: {'n_estimators': 1190, 'max_depth': 7, 'learning_rate': 0.021769858052516505, 'subsample': 0.8127204468230439, 'colsample_bytree': 0.9306358872951841, 'colsample_bylevel': 0.6120457392598307, 'colsample_bynode': 0.778270129945942, 'reg_alpha': 0.004038136910660049, 'reg_lambda': 0.007819555901792572, 'gamma': 0.43058356228734834, 'min_child_weight': 3, 'grow_policy': 'lossguide', 'max_leaves': 75, 'objective': 'reg:squarederror'}. Best is trial 48 with value: 0.8659439589259709.


Best trial: 48. Best value: 0.865944: 100%|██████████| 50/50 [04:24<00:00,  5.28s/it]

[I 2026-07-14 16:41:43,117] Trial 49 finished with value: 0.8137792019857631 and parameters: {'n_estimators': 1224, 'max_depth': 7, 'learning_rate': 0.024200108107430247, 'subsample': 0.58830701984194, 'colsample_bytree': 0.9041488595752516, 'colsample_bylevel': 0.6129655479691586, 'colsample_bynode': 0.8231407488467556, 'reg_alpha': 0.0027209511120594556, 'reg_lambda': 0.05310213882955393, 'gamma': 0.21853559076004314, 'min_child_weight': 1, 'grow_policy': 'lossguide', 'max_leaves': 65, 'objective': 'reg:pseudohubererror'}. Best is trial 48 with value: 0.8659439589259709.
Best CV R2: 0.8659439589259709
Best params: {'n_estimators': 1190, 'max_depth': 7, 'learning_rate': 0.021769858052516505, 'subsample': 0.8127204468230439, 'colsample_bytree': 0.9306358872951841, 'colsample_bylevel': 0.6120457392598307, 'colsample_bynode': 0.778270129945942, 'reg_alpha': 0.004038136910660049, 'reg_lambda': 0.007819555901792572, 'gamma': 0.43058356228734834, 'min_child_weight': 3, 'grow_policy': 'lossg

In [ ]:
# ============================================================
# Optuna优化完成后
# 使用最佳参数重新进行5折交叉验证
# 同时计算R²、RMSE和MAE
# ============================================================

from sklearn.model_selection import cross_validate
import numpy as np


# ============================================================
# 1. 获取最佳参数
# ============================================================
best_params = study.best_params.copy()

best_params.update({
    "objective": "reg:squarederror",
    "random_state": 72,
    "n_jobs": 1
})


# ============================================================
# 2. 使用最佳参数构建模型
# ============================================================
best_cv_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(**best_params))
])


# ============================================================
# 3. 定义评价指标
# ============================================================
scoring = {
    "R2": "r2",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error"
}

cv = KFold(
        n_splits=10,
        shuffle=True,
        random_state=72
    )

# ============================================================
# 4. 进行10折交叉验证
# ============================================================
cv_results = cross_validate(
    best_cv_model,
    X_train_raw,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False
)


# ============================================================
# 5. 提取各折性能
# ============================================================
r2_scores = cv_results["test_R2"]

rmse_scores = -cv_results["test_RMSE"]

mae_scores = -cv_results["test_MAE"]


# ============================================================
# 6. 打印每一折性能
# ============================================================
print("\n" + "=" * 75)
print("Best model - 5-fold cross-validation performance")
print("=" * 75)

for fold in range(len(r2_scores)):

    print(
        f"Fold {fold + 1}: "
        f"R² = {r2_scores[fold]:.4f}, "
        f"RMSE = {rmse_scores[fold]:.4f}, "
        f"MAE = {mae_scores[fold]:.4f}"
    )


# ============================================================
# 7. 打印平均值 ± 标准差
# ============================================================
print("\nMean cross-validation performance:")

print(
    f"R²   = {np.mean(r2_scores):.4f} "
    f"± {np.std(r2_scores):.4f}"
)

print(
    f"RMSE = {np.mean(rmse_scores):.4f} "
    f"± {np.std(rmse_scores):.4f}"
)

print(
    f"MAE  = {np.mean(mae_scores):.4f} "
    f"± {np.std(mae_scores):.4f}"
)

print("=" * 75)


Best model - 5-fold cross-validation performance
Fold 1: R² = 0.9199, RMSE = 5.1564, MAE = 3.3497
Fold 2: R² = 0.8570, RMSE = 7.0092, MAE = 3.7668
Fold 3: R² = 0.8352, RMSE = 5.5920, MAE = 3.2465
Fold 4: R² = 0.8300, RMSE = 7.9772, MAE = 4.3214
Fold 5: R² = 0.8746, RMSE = 6.1907, MAE = 3.2545
Fold 6: R² = 0.8752, RMSE = 5.7913, MAE = 3.1462
Fold 7: R² = 0.8670, RMSE = 5.7210, MAE = 3.4693
Fold 8: R² = 0.8975, RMSE = 6.0817, MAE = 3.4300
Fold 9: R² = 0.8210, RMSE = 6.1666, MAE = 3.3560
Fold 10: R² = 0.8821, RMSE = 5.6219, MAE = 3.0626

Mean cross-validation performance:
R²   = 0.8659 ± 0.0295
RMSE = 6.1308 ± 0.7726
MAE  = 3.4403 ± 0.3465


In [28]:
# =========================
# 7. 使用最佳参数训练最终模型
# =========================
best_params = study.best_params.copy()

best_params.update({
    "objective": "reg:squarederror",
    "random_state": 72,
    "n_jobs": 1
})

final_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(**best_params))
])

final_model.fit(X_train_raw, y_train)


# =========================
# 8. 测试集预测
# =========================
y_train_pred = final_model.predict(X_train_raw)
y_test_pred = final_model.predict(X_test_raw)

# =========================
# 9. 模型评价
# =========================
def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"\n{name} performance:")
    print(f"R2   = {r2:.4f}")
    print(f"RMSE = {rmse:.4f}")
    print(f"MAE  = {mae:.4f}")

    return r2, rmse, mae


train_r2, train_rmse, train_mae = evaluate(y_train, y_train_pred, "Train")
test_r2, test_rmse, test_mae = evaluate(y_test, y_test_pred, "Test")


Train performance:
R2   = 0.9935
RMSE = 1.3738
MAE  = 0.8916

Test performance:
R2   = 0.9132
RMSE = 4.9467
MAE  = 3.0087


In [29]:
# =========================
# 10. 导出原始训练集/测试集
# =========================
train_raw = X_train_raw.copy()
test_raw = X_test_raw.copy()

train_raw[target_col] = y_train.values
test_raw[target_col] = y_test.values

train_raw.to_excel("train_raw.xlsx", index=False)
test_raw.to_excel("test_raw.xlsx", index=False)


# =========================
# 11. 导出归一化后的训练集/测试集
# 注意：这里的 scaler 只在训练集上 fit
# =========================
fitted_preprocessor = final_model.named_steps["preprocess"]

X_train_scaled_array = fitted_preprocessor.transform(X_train_raw)
X_test_scaled_array = fitted_preprocessor.transform(X_test_raw)

scaled_feature_names = fitted_preprocessor.get_feature_names_out()

X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=scaled_feature_names,
    index=X_train_raw.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=scaled_feature_names,
    index=X_test_raw.index
)

train_scaled = X_train_scaled.copy()
test_scaled = X_test_scaled.copy()

train_scaled[target_col] = y_train.values
test_scaled[target_col] = y_test.values

train_scaled.to_excel("train_scaled.xlsx", index=False)
test_scaled.to_excel("test_scaled.xlsx", index=False)

In [30]:
# =========================
# 12. 导出预测结果
# =========================
train_pred_df = pd.DataFrame({
    "y_true": y_train.values,
    "y_pred": y_train_pred
})

test_pred_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": y_test_pred
})

train_pred_df.to_excel("train_prediction.xlsx", index=False)
test_pred_df.to_excel("test_prediction.xlsx", index=False)


# =========================
# 13. 保存模型、参数和归一化列
# =========================
joblib.dump(final_model, "final_CDHLQ.pkl")

with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

with open("scaler_columns.json", "w") as f:
    json.dump(numerical_cols_to_scale, f, indent=4)


# 打印最优超参数
print("\n==============================")
print("Best Hyperparameters")
print("==============================")
for key, value in best_params.items():
    print(f"{key}: {value}")

print("\nBest 10-fold CV R²: {:.4f}".format(study.best_value))


Best Hyperparameters
n_estimators: 1190
max_depth: 7
learning_rate: 0.021769858052516505
subsample: 0.8127204468230439
colsample_bytree: 0.9306358872951841
colsample_bylevel: 0.6120457392598307
colsample_bynode: 0.778270129945942
reg_alpha: 0.004038136910660049
reg_lambda: 0.007819555901792572
gamma: 0.43058356228734834
min_child_weight: 3
grow_policy: lossguide
max_leaves: 75
objective: reg:squarederror
random_state: 72
n_jobs: 1

Best 10-fold CV R²: 0.8659


In [32]:
from collections import defaultdict
import random

# =========================
# 随机生成20个互不重复的随机种子
# =========================
random.seed(72)   # 固定生成的随机种子，保证可复现
random_states = random.sample(range(1, 500), 20)

print("Random states:")
print(random_states)

results = defaultdict(list)

for run, rs in enumerate(random_states, start=1):

    print(f"\nRun {run}/20 (random_state={rs})")

    # 每次重新随机划分
    X_train_i_raw, X_test_i_raw, y_train_i, y_test_i = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=rs
    )

    # 构建Pipeline
    model_i = Pipeline([
        ("preprocess", ColumnTransformer(
            transformers=[
                ("scale", StandardScaler(), numerical_cols_to_scale)
            ],
            remainder="passthrough"
        )),
        ("model", XGBRegressor(**best_params))
    ])

    # 训练
    model_i.fit(X_train_i_raw, y_train_i)

    # 预测
    y_train_pred_i = model_i.predict(X_train_i_raw)
    y_test_pred_i = model_i.predict(X_test_i_raw)

    # 指标
    train_r2_i = r2_score(y_train_i, y_train_pred_i)
    train_mae_i = mean_absolute_error(y_train_i, y_train_pred_i)
    train_rmse_i = np.sqrt(mean_squared_error(y_train_i, y_train_pred_i))

    test_r2_i = r2_score(y_test_i, y_test_pred_i)
    test_mae_i = mean_absolute_error(y_test_i, y_test_pred_i)
    test_rmse_i = np.sqrt(mean_squared_error(y_test_i, y_test_pred_i))

    # 保存
    results["Random_State"].append(rs)

    results["Train_R2"].append(train_r2_i)
    results["Train_MAE"].append(train_mae_i)
    results["Train_RMSE"].append(train_rmse_i)

    results["Test_R2"].append(test_r2_i)
    results["Test_MAE"].append(test_mae_i)
    results["Test_RMSE"].append(test_rmse_i)

# =========================
# 汇总结果
# =========================
results_df = pd.DataFrame(results)
results_df.index = [f"Run_{i + 1}" for i in range(20)]

# 添加均值和标准差
results_df.loc["Mean"] = results_df.mean(numeric_only=True)
results_df.loc["Std"] = results_df.std(numeric_only=True)

# Mean和Std没有对应的随机种子
results_df.loc["Mean", "Random_State"] = ""
results_df.loc["Std", "Random_State"] = ""

# 导出
results_df.to_excel("CDHLQ_20_random_splits.xlsx")

print("\n✅ 20 次随机划分验证完成，结果已保存至：CDHLQ_20_random_splits.xlsx")

Random states:
[38, 401, 305, 379, 96, 177, 358, 279, 318, 192, 158, 368, 351, 307, 67, 360, 356, 176, 120, 195]

Run 1/20 (random_state=38)

Run 2/20 (random_state=401)

Run 3/20 (random_state=305)

Run 4/20 (random_state=379)

Run 5/20 (random_state=96)

Run 6/20 (random_state=177)

Run 7/20 (random_state=358)

Run 8/20 (random_state=279)

Run 9/20 (random_state=318)

Run 10/20 (random_state=192)

Run 11/20 (random_state=158)

Run 12/20 (random_state=368)

Run 13/20 (random_state=351)

Run 14/20 (random_state=307)

Run 15/20 (random_state=67)

Run 16/20 (random_state=360)

Run 17/20 (random_state=356)

Run 18/20 (random_state=176)

Run 19/20 (random_state=120)

Run 20/20 (random_state=195)

✅ 20 次随机划分验证完成，结果已保存至：CDHLQ_20_random_splits.xlsx


C:\Users\Peng Liu\AppData\Local\Temp\ipykernel_48288\2560796444.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  results_df.loc["Mean", "Random_State"] = ""
